In [2]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers
import pandas as pd

In [3]:

df = pd.DataFrame({
    "soil_moisture": [0.10, 0.15, 0.20, 0.25, 0.40, 0.60, 0.35, 0.18,
                      0.45, 0.05, 0.80, 0.27, 0.55, 0.70, 0.12, 0.30],
    "temperature_c": [34, 30, 26, 22, 28, 30, 19, 22,
                      35, 24, 33, 33, 21, 25, 20, 29],
    "sunlight_hours": [9, 8, 7, 4, 8, 10, 3, 10,
                       12, 5, 9, 11, 2, 6, 1, 9],
    "needs_water": [1, 1, 1, 0, 0, 0, 0, 1,
                    0, 1, 0, 1, 0, 0, 1, 1]
})

In [4]:
df

,soil_moisture,temperature_c,sunlight_hours,needs_water
0,0.10,34,9,1
1,0.15,30,8,1
2,0.20,26,7,1
3,0.25,22,4,0
4,0.40,28,8,0
5,0.60,30,10,0
6,0.35,19,3,0
7,0.18,22,10,1
8,0.45,35,12,0
9,0.05,24,5,1


In [5]:
df.columns

Index(['soil_moisture', 'temperature_c', 'sunlight_hours', 'needs_water'], dtype='object')

In [6]:
X = df[['soil_moisture', 'temperature_c', 'sunlight_hours']]
y = df['needs_water']

Normalisation in DL

In [7]:
X_min = X.min()
X_max = X.max()
X_scaled = (X - X_min) / (X_max - X_min + 1e-8)

In [8]:
X_scaled

,soil_moisture,temperature_c,sunlight_hours
0,0.066667,0.9375,0.727273
1,0.133333,0.6875,0.636364
2,0.200000,0.4375,0.545455
3,0.266667,0.1875,0.272727
4,0.466667,0.5625,0.636364
5,0.733333,0.6875,0.818182
6,0.400000,0.0000,0.181818
7,0.173333,0.1875,0.818182
8,0.533333,1.0000,1.000000
9,0.000000,0.3125,0.363636


In [9]:
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.25, random_state=42, stratify=y)

### In simple words

> `stratify=y` ensures that the **class distribution of `y` remains approximately the same** in both the training and testing datasets.

### Why is it important?

It is particularly useful when the dataset is **imbalanced**.

For example, if:

```text
Class 0 → 95%
Class 1 → 5%
```

we want both training and testing datasets to contain roughly:

```text
Class 0 → 95%
Class 1 → 5%
```

This helps the model get a representative training set and makes the test set a better representation of the original data.

### Key Point

```python
stratify=y
```

means:

**"Split the data while preserving the class proportions present in `y`."**


In [10]:
model = keras.Sequential([
    layers.Input(shape= (X_train.shape[1],)),
    layers.Dense(8, activation='relu'), # hidden layer - layers.Dense(power of 2, activation='relu'),
    layers.Dense(1, activation='sigmoid'), # output layer -
])

## **Neural Network Model**

```python
model = keras.Sequential([
    layers.Input(shape=(X_train.shape[1],)),

    layers.Dense(8, activation='relu'),  # Hidden layer

    layers.Dense(1, activation='sigmoid'),  # Output layer
])
```

### Brief Explanation

* **`keras.Sequential`** → Creates a neural network where layers are arranged **one after another**.
* **`Input(shape=(X_train.shape[1],))`** → Defines the number of **input features**.
* **`Dense(8, activation='relu')`** → Hidden layer with **8 neurons** using the **ReLU** activation function.
* **`Dense(1, activation='sigmoid')`** → Output layer with **1 neuron** using **Sigmoid**, which gives a probability between **0 and 1**.

### Architecture

```text
Input Features
      ↓
8 Neurons (ReLU)
      ↓
1 Neuron (Sigmoid)
      ↓
Prediction
```

> **Note:** `8` is the number of neurons in the hidden layer. It is a design choice and can be changed depending on the problem.


In [11]:
model.compile(optimizer='sgd', loss='binary_crossentropy', metrics=['accuracy'])

## `model.compile()`

```python
model.compile(
    optimizer='sgd',
    loss='binary_crossentropy',
    metrics=['accuracy']
)
```

`model.compile()` **configures the neural network before training**. It tells the model **how to learn and how to measure its performance**.

### 1. `optimizer='sgd'`

**SGD = Stochastic Gradient Descent**

* Decides **how the model updates its weights**.
* Uses the **gradient of the loss** to move weights in a direction that reduces the error.

```text
Prediction
    ↓
Calculate Loss
    ↓
Calculate Gradients
    ↓
SGD updates weights
    ↓
Better Prediction
```

### 2. `loss='binary_crossentropy'`

The **loss function** measures how wrong the model's predictions are.

`binary_crossentropy` is used for **binary classification**, where there are two classes:

```text
0 → Negative
1 → Positive
```

It works well with the `sigmoid` activation in the output layer.

```python
layers.Dense(1, activation='sigmoid')
```

The sigmoid produces a probability between **0 and 1**, and binary cross-entropy measures how close that probability is to the actual label.

### 3. `metrics=['accuracy']`

`accuracy` tells Keras to **calculate the percentage of correct predictions** during training and testing.

For example:

```text
100 predictions
80 correct
↓
Accuracy = 80%
```

### In short

| Parameter                    | Purpose                        |
| ---------------------------- | ------------------------------ |
| `optimizer='sgd'`            | **How to update weights**      |
| `loss='binary_crossentropy'` | **How to calculate error**     |
| `metrics=['accuracy']`       | **How to measure performance** |

### Simple way to remember

> **Optimizer → How to learn**
> **Loss → How wrong are we?**
> **Metric → How well are we doing?**


In [15]:
history_full = model.fit(X_train, y_train, epochs=100, batch_size=len(X_train), verbose=1) # send all at one time

Epoch 1/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step - accuracy: 1.0000 - loss: 0.5570
Epoch 2/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 109ms/step - accuracy: 1.0000 - loss: 0.5567
Epoch 3/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - accuracy: 1.0000 - loss: 0.5564
Epoch 4/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 1.0000 - loss: 0.5561
Epoch 5/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - accuracy: 1.0000 - loss: 0.5558
Epoch 6/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 1.0000 - loss: 0.5555
Epoch 7/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - accuracy: 1.0000 - loss: 0.5552
Epoch 8/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - accuracy: 1.0000 - loss: 0.5549
Epoch 9/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 1.0000 - loss: 0.5546
Epoch 10/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - accuracy: 1.0000 - loss: 0.5543
Epoch 11/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - accuracy: 1.0000 - loss: 0.5540
Epoch 12/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 1.0000 - l

In [17]:
X_train_np = np.asarray(X_train, dtype=np.float32)
X_test_np  = np.asarray(X_test, dtype=np.float32)

y_train_np = np.asarray(y_train, dtype=np.float32)
y_test_np  = np.asarray(y_test, dtype=np.float32)

history = model.fit(
    X_train_np,
    y_train_np,
    validation_data=(X_test_np, y_test_np),
    epochs=100,
    batch_size=1, # one at a time
    verbose=1
)

Epoch 1/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 35ms/step - accuracy: 1.0000 - loss: 0.4528 - val_accuracy: 1.0000 - val_loss: 0.4869
Epoch 2/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9167 - loss: 0.4503 - val_accuracy: 1.0000 - val_loss: 0.4844
Epoch 3/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9167 - loss: 0.4477 - val_accuracy: 1.0000 - val_loss: 0.4809
Epoch 4/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9167 - loss: 0.4449 - val_accuracy: 1.0000 - val_loss: 0.4781
Epoch 5/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9167 - loss: 0.4421 - val_accuracy: 1.0000 - val_loss: 0.4752
Epoch 6/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9167 - loss: 0.4393 - val_accuracy: 1.0000 - val_loss: 0.4724
Epoch 7/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9167 - loss: 0.4366 - val_accuracy: 1.0000 - val_loss: 0.4691
Epoch 8/100
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9167 - loss: 0.4339 - val_accuracy: 1.0000 -

## Converting Data and Training the Model

### Code

X_train_np = np.asarray(X_train, dtype=np.float32)
X_test_np  = np.asarray(X_test, dtype=np.float32)

y_train_np = np.asarray(y_train, dtype=np.float32)
y_test_np  = np.asarray(y_test, dtype=np.float32)

history = model.fit(
    X_train_np,
    y_train_np,
    validation_data=(X_test_np, y_test_np),
    epochs=100,
    batch_size=4,
    verbose=1
)

### Explanation

- **`np.asarray()`** → Converts the data into NumPy arrays.
- **`dtype=np.float32`** → Converts values to `float32` for TensorFlow/Keras.
- **`X_train_np, y_train_np`** → Training features and labels.
- **`validation_data`** → Evaluates the model using test data.
- **`epochs=100`** → Trains the model 100 times on the complete training data.
- **`batch_size=4`** → Processes 4 samples at a time before updating weights (if there are 10 rows then send them all instead of sending a single row).
- **`verbose=1`** → Displays training progress.
- **`history`** → Stores training and validation loss/accuracy.

### In Short

**Convert data → Train → Validate → Repeat 100 times → Store results**

## Cost Function vs Mean Squared Error (MSE)

### What is a Cost Function?

A **cost function** measures how wrong the model's predictions are compared to the actual values.

It gives us a single number representing the **overall error** of the model.

> **Goal of training:** Minimize the cost function so that the model's predictions become better.

### Example

Suppose the actual values are:

Actual: `[10, 20, 30]`

Predicted: `[12, 18, 35]`

The cost function calculates how different the predictions are from the actual values.

Loss function — error for a single training example

Cost function — error averaged/summed over the entire dataset (or a batch)

---

## What is Mean Squared Error (MSE)?

**MSE is one type of cost/loss function.**

Formula:

$$
MSE = \frac{1}{n}\sum_{i=1}^{n}(y_i-\hat{y}_i)^2
$$

Where:

- $y_i$ = Actual value
- $\hat{y}_i$ = Predicted value
- $n$ = Number of samples

### Example

Actual: `[10, 20]`

Predicted: `[12, 18]`

```text
Errors:
10 - 12 = -2
20 - 18 =  2

Squared errors:
(-2)² = 4
(2)²  = 4

MSE = (4 + 4) / 2
    = 4

## MAE (Mean Absolute Error)

MAE is another cost function, closely related to MSE but with one key difference: it uses absolute differences instead of squared differences.

$$MAE = \frac{1}{n}\sum_{i=1}^{n}|y_i - \hat{y}_i|$$

## MSE vs MAE — Core Differences

| | MSE | MAE |
|---|---|---|
| Formula | Squares the error | Takes absolute value of error |
| Sensitivity to outliers | High — big errors get squared, so they dominate | Low — treats all errors linearly |
| Gradient behavior | Smooth, differentiable everywhere; gradient scales with error size | Gradient is constant (±1) regardless of error size; not differentiable at 0 |
| Penalizes | Large errors much more heavily | All errors proportionally |
| Typical use case | When large errors are especially bad (and data doesn't have many outliers) | When you have outliers you don't want to dominate training, or want robustness |

## Why the Outlier Behavior Matters in Practice

- Predicted 10, actual 11 → error = 1 → MSE contributes 1, MAE contributes 1 (same)
- Predicted 10, actual 20 → error = 10 → MSE contributes 100, MAE contributes 10 (MSE punishes this way more)

MSE pushes your model harder to fix large errors, sometimes at the cost of many small errors elsewhere. MAE treats all mistakes "fairly," but the non-differentiability at zero can make gradient-based optimization trickier — often needs subgradient methods or smoothing tricks.

## Where Huber Loss Fits In

Huber Loss combines both: behaves like MSE for small errors (smooth gradient) and like MAE for large errors (robust to outliers). Useful when pure MSE is too outlier-sensitive but pure MAE is too unstable to train.

## MSLE (Mean Squared Logarithmic Error) - Mainly used when there is growth to find

MSLE applies a log transform to predictions and actual values before squaring the difference:

$$MSLE = \frac{1}{n}\sum_{i=1}^{n}(\log(y_i + 1) - \log(\hat{y}_i + 1))^2$$

The "+1" (sometimes written log1p) avoids taking log(0), which is undefined.

## Why the Log Transform Matters

- It penalizes **relative** (percentage) errors rather than **absolute** errors.
- Under-predictions and over-predictions of the same *ratio* are penalized similarly, unlike MSE where the actual magnitude of the difference dominates.
- Example: actual = 10, predicted = 20 (off by 10, ratio 2x) vs actual = 1000, predicted = 1010 (off by 10, ratio ~1x) — MSE treats both errors as equally bad (squared diff = 100 in both cases), but MSLE recognizes the first case is proportionally much worse.

## MSLE vs MSE

| | MSE | MSLE |
|---|---|---|
| Error type measured | Absolute difference | Relative / percentage difference |
| Sensitivity to large values | High — large absolute errors dominate | Reduced — large values are compressed by the log |
| Sensitivity to outliers | High | Lower, since log shrinks the scale of big numbers |
| Penalizes under vs over prediction | Symmetric | **Asymmetric** — penalizes under-prediction more heavily than over-prediction |
| Typical use case | General regression | Targets that grow exponentially or span several orders of magnitude (e.g. population, sales, counts) — where you care about relative error, not absolute error |

## Important Caveat: Requires Non-Negative Values

Since it takes a log, MSLE assumes \(y_i, \hat{y}_i \geq 0\). It breaks down (or needs modification) if your targets can be negative.

## The Asymmetry, Explained

For a fixed absolute gap, under-predicting is penalized more than over-predicting:

- Actual = 100, predicted = 50 (under by 50) → larger log-ratio penalty
- Actual = 100, predicted = 150 (over by 50) → smaller log-ratio penalty

This makes MSLE a natural fit when under-forecasting is more costly than over-forecasting (e.g. demand forecasting, where running out of stock is worse than having excess).